# 춘천시 아동급식카드 가맹점 전처리 2차

**Google Colab 실행 노트북**

가맹점 데이터 정리 · 업종 검토 · 학교 주변 상가 분석 · 좌표 연결

**실행 환경:** Google Colab과 Google Drive (`MyDrive/data/`에 입력 데이터 준비)

**작업 순서:**

1단계부터 9단계까지: 기본 전처리

10단계: 통합 엑셀 생성

11단계부터 16단계까지: 좌표 확인 및 추가 연결

**결과 저장 위치:** `MyDrive/data/processed_v2/`

**통합 엑셀:** `MyDrive/data/데이터톤_전처리_통합.xlsx`

코드 셀은 **위에서 아래 순서대로 실행**한다. 원본 CSV와 Google Drive 경로가 준비되어 있어야 한다. 분석 결과에 포함된 의심·추정·보류 상태는 사람의 최종 검토 결과와 구분한다.



**목표:** 춘천시 아동급식카드 가맹점 데이터를 기준으로 (A) 지도 표시용 품질 점검, (B) 학교 주변 사각지대 분석용 위치 데이터를 준비한다.

### 데이터 처리 원칙
- **원본 6종은 수정·삭제하지 않는다.** 결과는 `data/processed_v2/` 아래에 별도로 저장한다.
- 이름 띄어쓰기를 통일하되, **이름이 같다는 이유만으로 가게를 합치지 않는다.** 동일 상호+동일 건물은 중복 **의심**으로 분리한다.
- `기타`, `일반대중음식`, `호프/통닭`만으로 부적합한 가맹점이라고 확정하지 않는다.
- **주류 판매가 주된 목적**인지 사람의 확인이 필요하다. 검증 전에는 `검토보류`로 표시한다.
- 학교 주변 사각지대를 단정하려면 **현재 가맹점의 신뢰할 수 있는 좌표**가 필요하다. B에서는 먼저 학교·상가 위치 자료와 비교용 목록을 준비한다.

**기존 결과와의 관계:** 이전의 2,879개 매칭 후보와 197개 주점 관련 업태 후보는 검토 참고 자료로 활용한다. 이번 결과는 개수가 달라질 수 있으므로 이전 수치를 강제로 맞추지 않는다.

## 1. Google Drive 연결 및 데이터 불러오기

아래 코드가 실제로 읽는 주요 원본은 급식카드 가맹점과 요식업소 현황이다. B용으로 상가정보와 학교 위치 파일도 불러온다.
원본 파일명이 한글 자모 분리(NFD) 형태여도 찾을 수 있게 처리했다.

In [ ]:
import os
import re
import unicodedata
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/data'
OUT = os.path.join(DATA_DIR, 'processed_v2')
A_DIR = os.path.join(OUT, 'A_지도')
B_DIR = os.path.join(OUT, 'B_사각지대')
os.makedirs(A_DIR, exist_ok=True)
os.makedirs(B_DIR, exist_ok=True)

def find_csv(filename):
    expected = unicodedata.normalize('NFC', filename)
    for file in os.listdir(DATA_DIR):
        if unicodedata.normalize('NFC', file) == expected:
            return os.path.join(DATA_DIR, file)
    raise FileNotFoundError('원본 CSV를 찾을 수 없음: ' + filename)

def read_csv(filename):
    path = find_csv(filename)
    try:
        return pd.read_csv(path, encoding='utf-8-sig', low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='cp949', low_memory=False)

def save_csv(data, path):
    data.to_csv(path, index=False, encoding='utf-8-sig')
    print(f'저장: {os.path.basename(path)} ({len(data):,}행)')

card_raw = read_csv('강원특별자치도 춘천시_아동급식카드 가맹점_20260827.csv')
card_raw = card_raw.rename(columns={'기맹점명':'가맹점명'})
restaurant_raw = read_csv('강원특별자치도 춘천시_요식업소 현황_20260527.csv')
store_raw = read_csv('소상공인시장진흥공단_상가(상권)정보_강원_202606.csv')
school_raw = read_csv('전국초중등학교위치표준데이터.csv')

print('급식카드 가맹점:', card_raw.shape)
print('요식업소 현황:', restaurant_raw.shape)
print('강원 상가정보:', store_raw.shape)
print('학교 위치:', school_raw.shape)
print('상가 좌표 컬럼 후보:', [c for c in store_raw.columns if '위도' in c or '경도' in c])
print('학교 좌표 컬럼 후보:', [c for c in school_raw.columns if '위도' in c or '경도' in c])

Mounted at /content/drive
급식카드 가맹점: (4593, 10)
요식업소 현황: (6184, 7)
강원 상가정보: (110206, 39)
학교 위치: (12004, 20)
상가 좌표 컬럼 후보: ['경도', '위도']
학교 좌표 컬럼 후보: ['위도', '경도']


## 2. 이름·주소 정규화와 완전 중복 처리

띄어쓰기, 대소문자, 지역 명칭 표기를 비교용 컬럼에서 통일한다. **실제 가맹점명과 주소 원본 컬럼은 변경하지 않는다.**
동일한 모든 컬럼을 가진 행은 완전 중복으로 기록한 후 제거하지만, 업종이 다른 행은 그대로 남긴다.

In [ ]:
def norm_name(value):
    if pd.isna(value):
        return ''
    value = unicodedata.normalize('NFKC', str(value)).casefold().strip()
    return re.sub(r'\s+', '', value)

def norm_address(value):
    if pd.isna(value):
        return ''
    value = unicodedata.normalize('NFKC', str(value)).strip()
    value = value.replace('강원특별자치도', '강원도')
    value = re.sub(r'^강원(?=\s+춘천시)', '강원도', value)
    value = re.sub(r'\s+', ' ', value)
    return value.strip()

def road_key(value):
    value = norm_address(value)
    # 도로명 + 건물번호. 호실/층수는 포함하지 않는다.
    m = re.search(r'([가-힣A-Za-z0-9·.-]+(?:대로|로|길))\s*(\d+(?:-\d+)?)\b', value)
    return m.group(1) + ' ' + m.group(2) if m else ''

def jibun_key(value):
    value = norm_address(value)
    m = re.search(r'([가-힣0-9]+(?:동|리))\s*(산\s*)?(\d+)(?:\s*-\s*(\d+))?\s*(?:번지)?', value)
    if not m:
        return ''
    return m.group(1) + ' ' + ('산' if m.group(2) else '') + m.group(3) + (('-' + m.group(4)) if m.group(4) else '')

card_raw = card_raw.reset_index(drop=True)
card_raw['원본행번호'] = card_raw.index
# 원본행번호를 제외한 원본 데이터의 모든 컬럼으로 완전 중복 판정
original_cols = [c for c in card_raw.columns if c != '원본행번호']
removed = card_raw[card_raw.duplicated(subset=original_cols, keep='first')].copy()
card = card_raw.drop_duplicates(subset=original_cols, keep='first').copy()

card['가맹점ID'] = card['원본행번호']
card['이름키'] = card['가맹점명'].apply(norm_name)
card['주소정리'] = card['소재지도로명주소'].apply(norm_address)
card['도로명키'] = card['소재지도로명주소'].apply(road_key)
card['지번키'] = card['소재지도로명주소'].apply(jibun_key)
# 비교용 전체주소키: 공백 차이는 무시하지만 101호와 102호 등은 구별
card['상세주소키'] = card['주소정리'].apply(lambda s: re.sub(r'\s+', '', s))

print('원본:', len(card_raw), '완전 중복 제거:', len(removed), '정리 후:', len(card))
save_csv(removed, os.path.join(A_DIR, 'A_완전중복_제거기록.csv'))

원본: 4593 완전 중복 제거: 25 정리 후: 4568
저장: A_완전중복_제거기록.csv (25행)


## 3. 동일 점포 의심 중복 찾기

같은 상호가 같은 건물에 여러 번 등장하면 검토 목록으로 보낸다.

- `씨유(CU) 춘천...`처럼 띄어쓰기가 다른 상호는 같은 이름키로 비교한다.
- `숙성21`처럼 이름·주소가 같더라도 업종이 다르면 **자동으로 삭제하지 않는다.**
- 같은 건물의 서로 다른 호실과 지점은 실제 다른 점포일 수 있다.
- 주소키를 추출하지 못한 행끼리 **빈 값 때문에 중복 판정되지 않도록** 한다.

In [ ]:
# 도로명주소가 있으면 건물 단위, 없으면 지번 단위, 둘 다 없으면 상세주소 단위로 확인
card['위치키'] = card.apply(
    lambda row: '도로:' + row['도로명키'] if row['도로명키'] else (
        '지번:' + row['지번키'] if row['지번키'] else (
            '문자:' + row['상세주소키'] if row['상세주소키'] else ''
        )
    ), axis=1
)
valid = (card['이름키'] != '') & (card['위치키'] != '')
card['중복의심'] = False
card.loc[valid, '중복의심'] = card.loc[valid].duplicated(
    subset=['이름키','위치키'], keep=False
)

# 완전 주소까지 비슷한지 보조 표시 (업종 차이 등은 직접 확인)
card['중복이유'] = ''
card.loc[card['중복의심'], '중복이유'] = '상호명+건물/지번 일치: 검토 필요'

suspect = card[card['중복의심']].sort_values(['이름키','위치키']).copy()
suspect['검토결과'] = ''
suspect['비고'] = ''
print('중복 의심 행:', len(suspect))
display(suspect[['가맹점ID','가맹점명','업종명','소재지도로명주소','이름키','위치키']].head(20))
save_csv(suspect, os.path.join(A_DIR,'A_중복의심_검토.csv'))

중복 의심 행: 350


,가맹점ID,가맹점명,업종명,소재지도로명주소,이름키,위치키
223,223,강원대학교생활협동조합,일반대중음식,강원도 춘천시 강원대학길 1 2층,강원대학교생활협동조합,도로:강원대학길 1
224,224,강원대학교생활협동조합,제과점,강원 춘천시 강원대학길 1 (효자동),강원대학교생활협동조합,도로:강원대학길 1
467,467,그집과일,청과물,강원도 춘천시 서부대성로 44-1 1층,그집과일,도로:서부대성로 44-1
468,468,그집과일,일반대중음식,강원 춘천시 서부대성로 44-1 (요선동),그집과일,도로:서부대성로 44-1
614,614,남춘천꼬마김밥,일반대중음식,강원도 춘천시 춘주로 90 1층 온의동,남춘천꼬마김밥,도로:춘주로 90
615,615,남춘천꼬마김밥,한식,강원특별자치도 춘천시 춘주로 90 1층,남춘천꼬마김밥,도로:춘주로 90
1144,1144,만천리상회,한식,강원특별자치도 춘천시 동면 금베이길 50 1층,만천리상회,도로:금베이길 50
1145,1145,만천리상회,일반대중음식,강원특별자치도 춘천시 동면 금베이길 50 .,만천리상회,도로:금베이길 50
1172,1172,맛있는생선요리집,한식,강원도 춘천시 보안길 41 1층,맛있는생선요리집,도로:보안길 41
1173,1173,맛있는생선요리집,한식,강원 춘천시 보안길 41 (후평동),맛있는생선요리집,도로:보안길 41


저장: A_중복의심_검토.csv (350행)


## 4. 요식업소 공식 업태 연결 및 '기타' 분류 점검

이전 전처리와 같은 매칭 규칙을 **원본 CSV에서 다시 실행**한다. 원본이 같다면 기존 매칭 수치(2,879개 후보)를 재현할 수 있다.

이후 원래 가맹점 분류와 공식 업태를 별도로 보존하고, `기타`처럼 뜻이 넓은 업종은 재분류 검토 목록으로 저장한다. **기타라고 부적합하다고 단정하지 않는다.**

In [ ]:
# 원본 CSV만으로 공식 업태 매칭 결과 재현 (이전 processed 파일이 없어도 실행됨)
restaurant_data = restaurant_raw.drop_duplicates().reset_index().rename(columns={'index':'restaurant_id'})
card_data = card.copy().rename(columns={'가맹점ID':'card_id'})

def match_name(value):
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', '', str(value).strip())

# 기존 노트북과 동일한 매칭 기준을 적용 (중복 감사용 이름키와 구분)
card_data['이름_정규화'] = card_data['가맹점명'].apply(match_name)
restaurant_data['이름_정규화'] = restaurant_data['업소명'].apply(match_name)
card_data['주소_정규화2'] = card_data['소재지도로명주소'].apply(norm_address)
restaurant_data['주소_정규화2'] = restaurant_data['소재지(도로명)'].apply(norm_address)

# 기존 코드와 동일한 도로명 및 지번 추출 함수 이름
get_road_address = road_key
get_jibun = jibun_key


keys = ["이름_정규화", "주소_정규화2"]
match_cols = ["card_id", "restaurant_id", "매칭방법", "매칭상태"]

# 1차: 상호명 + 전체 주소 일치
card_unique = card_data[
    (card_data["주소_정규화2"] != "") &
    ~card_data.duplicated(subset=keys, keep=False)
].copy()
restaurant_unique = restaurant_data[
    (restaurant_data["주소_정규화2"] != "") &
    ~restaurant_data.duplicated(subset=keys, keep=False)
].copy()

first = card_unique.merge(
    restaurant_unique, on=keys, how="inner",
    suffixes=("_card", "_restaurant"), validate="one_to_one"
)
first["매칭방법"] = "전체주소일치"
first["매칭상태"] = "1차매칭"

# 2차: 양쪽에서 상호명이 유일한 경우만 도로명+건물번호 비교
card_name_unique = card_data[
    ~card_data.duplicated(subset=["이름_정규화"], keep=False)
]
restaurant_name_unique = restaurant_data[
    ~restaurant_data.duplicated(subset=["이름_정규화"], keep=False)
]
second = card_name_unique.merge(
    restaurant_name_unique, on="이름_정규화", how="inner",
    suffixes=("_card", "_restaurant"), validate="one_to_one"
)
second = second[~second["card_id"].isin(first["card_id"])].copy()
second["도로명_card"] = second["주소_정규화2_card"].apply(get_road_address)
second["도로명_restaurant"] = second["주소_정규화2_restaurant"].apply(get_road_address)
second = second[
    (second["도로명_card"] != "") &
    (second["도로명_card"] == second["도로명_restaurant"])
].copy()
second["매칭방법"] = "기본주소일치"
second["매칭상태"] = "검증대기"

matches = pd.concat([first[match_cols], second[match_cols]], ignore_index=True)
assert matches["card_id"].is_unique
assert matches["restaurant_id"].is_unique

print("1차 전체주소 일치:", len(first))
print("2차 도로명+건물번호 일치 후보:", len(second))
print("1·2차 매칭 후보 합계:", len(matches))

# 기존 1·2차 매칭에서 미연결된 가맹점
unmatched = card_data[~card_data["card_id"].isin(matches["card_id"])].copy()

name_counts = restaurant_data["이름_정규화"].value_counts()
unmatched["동일상호_업소수"] = unmatched["이름_정규화"].map(name_counts).fillna(0).astype(int)

print("[미연결 상호명 분포]")
print("동일 상호명 없음:", (unmatched["동일상호_업소수"] == 0).sum())
print("동일 상호명 1개:", (unmatched["동일상호_업소수"] == 1).sum())
print("동일 상호명 2개 이상:", (unmatched["동일상호_업소수"] >= 2).sum())

# 이름이 유일한 요식업소를 주소 비교 대상으로 사용
restaurant_one = restaurant_data[
    ~restaurant_data.duplicated(subset=["이름_정규화"], keep=False)
][["restaurant_id", "이름_정규화", "주소_정규화2", "소재지(지번)"]]

target = unmatched[unmatched["동일상호_업소수"] == 1].copy()
check = target.merge(
    restaurant_one, on="이름_정규화", how="left",
    suffixes=("_card", "_restaurant"), validate="many_to_one"
)
check["도로명_card"] = check["주소_정규화2_card"].apply(get_road_address)
check["도로명_restaurant"] = check["주소_정규화2_restaurant"].apply(get_road_address)


def compare_road(row):
    a, b = row["도로명_card"], row["도로명_restaurant"]
    if not a or not b:
        return "도로명주소_추출불가"
    return "기본주소_일치" if a == b else "기본주소_불일치"

check["주소비교결과"] = check.apply(compare_road, axis=1)
print("\n[미연결 가맹점의 도로명주소 비교]")
print(check["주소비교결과"].value_counts())

# 5-1. 상호명 중복으로 제외됐던 기본주소 일치 후보
same_road = check[check["주소비교결과"] == "기본주소_일치"].copy()
same_road["이름주소_중복수"] = same_road.groupby(
    ["이름_정규화", "도로명_card"]
)["card_id"].transform("count")

road_extra = same_road[same_road["이름주소_중복수"] == 1][[
    "card_id", "restaurant_id", "이름_정규화"
]].copy()
road_extra["추가매칭방법"] = "기본주소_추가검토"

# 전체 가맹점에서 동일 이름+도로명키가 유일한지 재확인
card_road = card_data[["card_id", "이름_정규화", "주소_정규화2"]].copy()
card_road["도로명키"] = card_road["주소_정규화2"].apply(get_road_address)
card_road["전체_이름주소수"] = card_road.groupby(
    ["이름_정규화", "도로명키"]
)["card_id"].transform("count")
road_extra = road_extra.merge(
    card_road[["card_id", "전체_이름주소수"]], on="card_id", how="left"
)
road_extra = road_extra[road_extra["전체_이름주소수"] == 1][[
    "card_id", "restaurant_id", "추가매칭방법"
]]

# 5-2. 도로명주소를 추출할 수 없었던 후보의 지번 비교
jibun_check = check[check["주소비교결과"] == "도로명주소_추출불가"].copy()
jibun_check["가맹점_지번"] = jibun_check["소재지도로명주소"].apply(get_jibun)
jibun_check["요식업소_지번"] = jibun_check["소재지(지번)"].apply(get_jibun)

valid = (jibun_check["가맹점_지번"] != "") & (jibun_check["요식업소_지번"] != "")
same = valid & (jibun_check["가맹점_지번"] == jibun_check["요식업소_지번"])

jibun_check["지번비교결과"] = "확인불가"
jibun_check.loc[valid, "지번비교결과"] = "불일치"
jibun_check.loc[same, "지번비교결과"] = "일치후보"

jibun_extra = jibun_check[jibun_check["지번비교결과"] == "일치후보"][ [
    "card_id", "restaurant_id"
]].copy()
jibun_extra["추가매칭방법"] = "지번주소일치"

print("[지번주소 비교 결과]")
print(jibun_check["지번비교결과"].value_counts())

# 5-3. 중복 연결 및 충돌 후보 제외
extra = pd.concat([road_extra, jibun_extra], ignore_index=True)
extra = extra[
    ~extra["card_id"].isin(matches["card_id"]) &
    ~extra["restaurant_id"].isin(matches["restaurant_id"])
].copy()
extra = extra[~extra["restaurant_id"].duplicated(keep=False)].copy()

assert extra["card_id"].is_unique
assert extra["restaurant_id"].is_unique

print("\n[추가 후보]")
print("도로명 기본주소 단독 후보:", len(road_extra))
print("지번주소 일치 후보:", len(jibun_extra))
print("중복·충돌 제외 후:", len(extra))
print(extra["추가매칭방법"].value_counts())

# 추가 후보의 컬럼을 기존 매칭 결과와 통일
extra_matches = extra.rename(columns={"추가매칭방법": "매칭방법"}).copy()
extra_matches["매칭상태"] = "검증대기"
matches_v2 = pd.concat(
    [matches[match_cols], extra_matches[match_cols]], ignore_index=True
)

assert matches_v2["card_id"].is_unique
assert matches_v2["restaurant_id"].is_unique

# 원래 가맹점 전체에 공식 업태 추가
restaurant_info = restaurant_data[[
    "restaurant_id", "업소명", "업종명", "업태명", "소재지(도로명)"
]].rename(columns={
    "업종명": "공식_업종명",
    "업태명": "공식_업태명",
    "소재지(도로명)": "요식업소_주소"
})

final_v2 = card_data.merge(
    matches_v2, on="card_id", how="left", validate="one_to_one"
)
# 미연결 행은 restaurant_id가 여러 개 NaN이므로 many_to_one 검증
final_v2 = final_v2.merge(
    restaurant_info, on="restaurant_id", how="left", validate="many_to_one"
)
final_v2["매칭상태"] = final_v2["매칭상태"].fillna("미연결")
final_v2["매칭방법"] = final_v2["매칭방법"].fillna("없음")

assert len(final_v2) == len(card_data)
assert final_v2["card_id"].is_unique
assert final_v2["restaurant_id"].notna().sum() == len(matches_v2)

bar_types = ["정종/대포집/소주방", "라이브카페", "감성주점", "호프/통닭"]
bar_final = final_v2[final_v2["공식_업태명"].isin(bar_types)].copy()


def classify_type(value):
    if value in ["정종/대포집/소주방", "라이브카페", "감성주점"]:
        return "주점업태_우선점검"
    if value == "호프/통닭":
        return "호프통닭_추가분류"
    return "기타"

bar_final["검토그룹"] = bar_final["공식_업태명"].apply(classify_type)
bar_final["매칭검증결과"] = ""
bar_final["실제영업형태"] = ""
bar_final["비고"] = ""

print("[최종 데이터 현황]")
print("가맹점 전체:", len(final_v2))
print("매칭 및 후보:", len(matches_v2))
print("미연결:", (final_v2["매칭상태"] == "미연결").sum())
print("매칭 후보 비율:", round(len(matches_v2) / len(final_v2) * 100, 2), "%")
print("\n[매칭 방법]")
print(final_v2["매칭방법"].value_counts())
print("\n[주점 관련 공식 업태 후보]")
print(pd.crosstab(bar_final["공식_업태명"], bar_final["매칭상태"]))

# 원본 CSV가 동일한 버전이라면 아래 숫자와 비교
expected = {"가맹점": 4568, "매칭후보": 2879, "주점관련후보": 197}
actual = {"가맹점": len(final_v2), "매칭후보": len(matches_v2), "주점관련후보": len(bar_final)}
print("\n[이전 실행 결과와 비교]")
for key in expected:
    print(f"{key}: {actual[key]} (이전 {expected[key]})", "일치" if actual[key] == expected[key] else "차이 있음: 입력 데이터 및 매칭 조건 확인")

1차 전체주소 일치: 622
2차 도로명+건물번호 일치 후보: 2209
1·2차 매칭 후보 합계: 2831
[미연결 상호명 분포]
동일 상호명 없음: 1083
동일 상호명 1개: 535
동일 상호명 2개 이상: 119

[미연결 가맹점의 도로명주소 비교]
주소비교결과
기본주소_불일치      210
기본주소_일치       183
도로명주소_추출불가    142
Name: count, dtype: int64
[지번주소 비교 결과]
지번비교결과
불일치     63
일치후보    48
확인불가    31
Name: count, dtype: int64

[추가 후보]
도로명 기본주소 단독 후보: 8
지번주소 일치 후보: 48
중복·충돌 제외 후: 48
추가매칭방법
지번주소일치       40
기본주소_추가검토     8
Name: count, dtype: int64
[최종 데이터 현황]
가맹점 전체: 4568
매칭 및 후보: 2879
미연결: 1689
매칭 후보 비율: 63.03 %

[매칭 방법]
매칭방법
기본주소일치       2209
없음           1689
전체주소일치        622
지번주소일치         40
기본주소_추가검토       8
Name: count, dtype: int64

[주점 관련 공식 업태 후보]
매칭상태        1차매칭  검증대기
공식_업태명                
라이브카페          0     2
정종/대포집/소주방     7    40
호프/통닭         28   120

[이전 실행 결과와 비교]
가맹점: 4568 (이전 4568) 일치
매칭후보: 2879 (이전 2879) 일치
주점관련후보: 197 (이전 197) 일치


In [ ]:
# 매칭된 공식 업태는 참고용으로 연결: 미연결 행의 업태는 빈 값
card = card.merge(
    final_v2[['card_id','공식_업태명','매칭상태']].rename(columns={'card_id':'가맹점ID'}),
    on='가맹점ID', how='left', validate='one_to_one'
)

# 급식카드 원본 분류와 (있다면) 요식업소 분류를 각각 따로 보존
card['업종검토사유'] = ''
merchant_unknown = card['업종명'].fillna('').astype(str).str.strip().isin(['기타','미분류',''])
official_unknown = card['공식_업태명'].fillna('').astype(str).str.strip().isin(['기타','기타 휴게음식점'])
card.loc[merchant_unknown, '업종검토사유'] += '가맹점업종_기타/미분류;'
card.loc[official_unknown, '업종검토사유'] += '요식업소업태_기타;'

# 상호명으로 추측한 업종은 참고용이며 실제 업종 대체 금지
def hint_from_name(name):
    name = norm_name(name)
    if any(t in name for t in ['닭갈비','국밥','막국수','김밥','떡볶이','돈가스','식당','마라탕']):
        return '식사형_이름단서'
    if any(t in name for t in ['포차','주점','술집','바(bar)']):
        return '주점관련_이름단서'
    if any(t in name for t in ['치킨','통닭','닭강정']):
        return '치킨_이름단서'
    return ''
card['상호명_분류단서'] = card['가맹점명'].apply(hint_from_name)
category_review = card[card['업종검토사유'] != ''].copy()
category_review['확인된_실제업종'] = ''
category_review['근거'] = ''
print('업종 기타/미분류 검토행:',len(category_review))
save_csv(category_review, os.path.join(A_DIR, 'A_업종기타_재분류검토.csv'))

업종 기타/미분류 검토행: 414
저장: A_업종기타_재분류검토.csv (414행)


## 5. A: 주점 관련 업태 점검 및 지도용 임시 상태

'주류 판매가 주된 목적인 업소'인지 확인하는 것은 **업태 후보 추출 → 사람의 검수**로 진행한다. `호프/통닭`에는 치킨 식당이 섞여 있으므로 제외 확정하지 않는다.

**검토 상태의 의미**
- `주점관련_보류`: 공식 업태가 주점 관련이라 우선 확인 필요
- `업종불명_보류`: 업종 정보를 자세히 확인해야 함
- `참고업태미연결`: 요식업소 공식 업태가 아직 연결되지 않음
- `기초표시후보`: 위의 보류 사유가 없는 예비 후보. **적합성 확정은 아님**

**주점 의심이든 기타든 자동으로 삭제하지 않는다.** 사람 검수를 거쳐 `제외확정`된 가게만 최종 지도에서 제외한다.

In [ ]:
bar_priority = {'정종/대포집/소주방', '라이브카페', '감성주점'}
bar_mixed = {'호프/통닭'}

card['업태점검그룹'] = ''
card.loc[card['공식_업태명'].isin(bar_priority), '업태점검그룹'] = '주점업태_우선점검'
card.loc[card['공식_업태명'].isin(bar_mixed), '업태점검그룹'] = '호프통닭_추가분류'

# 기본적으로 검토 우선순위 표시. 공식 업태가 검증대기일 경우 분류 또한 잠정적임.
card['지도표시상태'] = '기초표시후보'
card.loc[card['중복의심'], '지도표시상태'] = '중복점포_검토보류'
card.loc[card['공식_업태명'].isna(), '지도표시상태'] = '참고업태미연결'
card.loc[card['업종검토사유'] != '', '지도표시상태'] = '업종불명_보류'
card.loc[card['업태점검그룹'] != '', '지도표시상태'] = '주점관련_보류'

bars = card[card['업태점검그룹'] != ''].copy()
bars['매칭확인'] = ''
bars['주류중심여부'] = ''
bars['실제식사가능여부'] = ''
bars['검수근거'] = ''

review_input = card[card['지도표시상태'] != '기초표시후보'][[
    '가맹점ID','가맹점명','소재지도로명주소','업종명','공식_업태명','매칭상태',
    '중복의심','업종검토사유','업태점검그룹','지도표시상태'
]].copy()
review_input['검수판정'] = ''  # 예: 표시가능 / 제외확정 / 보류
review_input['판정근거'] = ''
review_input['검수자'] = ''

print('지도 표시 상태별 개수:')
print(card['지도표시상태'].value_counts())
print('주점 관련 우선점검·호프통닭:',len(bars))
save_csv(card, os.path.join(A_DIR,'A_가맹점_전체정리.csv'))
save_csv(bars, os.path.join(A_DIR,'A_주점관련_검토후보.csv'))
save_csv(review_input, os.path.join(A_DIR,'A_사람검수_입력양식.csv'))
save_csv(card, os.path.join(A_DIR,'A_지도용_검토상태포함.csv'))

지도 표시 상태별 개수:
지도표시상태
기초표시후보       2240
참고업태미연결      1689
업종불명_보류       414
주점관련_보류       197
중복점포_검토보류      28
Name: count, dtype: int64
주점 관련 우선점검·호프통닭: 197
저장: A_가맹점_전체정리.csv (4,568행)
저장: A_주점관련_검토후보.csv (197행)
저장: A_사람검수_입력양식.csv (2,328행)
저장: A_지도용_검토상태포함.csv (4,568행)


## 6. B: 춘천시 학교와 상가 위치 자료 준비

현재 가맹점 목록은 **위도·경도가 없으므로**, 이 셀에서는 우선 학교와 상가의 좌표를 정리한다.
소상공인 상가정보에는 현재 가맹점이 이미 섞여 있을 수 있다. 정확한 가맹 여부를 재확인하기 전에는 '비가맹점'이라고 단정하지 않는다.

In [ ]:
def filter_chuncheon(data):
    # 현재 데이터에 실제 존재하는 컬럼 우선 사용
    if '시군구명' in data.columns:
        return data[data['시군구명'].fillna('').astype(str).str.contains('춘천시', regex=False)].copy()
    if '소재지도로명주소' in data.columns:
        return data[data['소재지도로명주소'].fillna('').astype(str).str.contains('춘천시',regex=False)].copy()
    if '도로명주소' in data.columns:
        return data[data['도로명주소'].fillna('').astype(str).str.contains('춘천시',regex=False)].copy()
    raise ValueError('춘천시를 식별할 컬럼이 없습니다. 컬럼 확인: ' + str(data.columns.tolist()))

def attach_coordinates(data):
    data = data.copy()
    lat_col = next((c for c in ['위도','위도값','latitude','lat','Latitude'] if c in data.columns), None)
    lon_col = next((c for c in ['경도','경도값','longitude','lon','Longitude'] if c in data.columns), None)
    data['위도_분석'] = pd.to_numeric(data[lat_col], errors='coerce') if lat_col else float('nan')
    data['경도_분석'] = pd.to_numeric(data[lon_col], errors='coerce') if lon_col else float('nan')
    return data

stores = attach_coordinates(filter_chuncheon(store_raw))
schools = attach_coordinates(filter_chuncheon(school_raw))

print('춘천 상가:', len(stores), '좌표 있는 행:', stores[['위도_분석','경도_분석']].notna().all(axis=1).sum())
print('춘천 학교:', len(schools), '좌표 있는 행:', schools[['위도_분석','경도_분석']].notna().all(axis=1).sum())
save_csv(stores, os.path.join(B_DIR,'B_춘천상가_좌표준비.csv'))
save_csv(schools, os.path.join(B_DIR,'B_춘천학교_좌표준비.csv'))

춘천 상가: 17984 좌표 있는 행: 17984
춘천 학교: 76 좌표 있는 행: 76
저장: B_춘천상가_좌표준비.csv (17,984행)
저장: B_춘천학교_좌표준비.csv (76행)


## 7. B: 가맹 여부 확인이 필요한 상가 후보

상가정보의 상호·주소와 현재 가맹점의 상호·주소를 비교한다.

- 상호명과 건물번호가 일치하면 `기존가맹점_추정`으로 표기한다.
- 일치하지 않는다고 해서 무조건 **비가맹점은 아니다.** 상호 변경, 주소 형식 차이, 누락 등이 있기 때문에 `가맹여부_확인필요`로 표기한다.
- 아직 학교 주변 사각지대 판정은 하지 않는다. 해당 판정에는 **현재 급식카드 가맹점 좌표의 충분한 확보와 검증**이 필요하다.

In [ ]:
# 상가정보의 상호명·주소 컬럼을 실제 존재하는 컬럼에서 선택
store_name_col = next((c for c in ['상호명','사업장명','업소명'] if c in stores.columns), None)
store_addr_col = next((c for c in ['도로명주소','소재지도로명주소','소재지(도로명)'] if c in stores.columns), None)
if not store_name_col or not store_addr_col:
    print('상가정보에 필요한 상호명/주소 컬럼을 못 찾음:', stores.columns.tolist())
else:
    stores['이름키'] = stores[store_name_col].apply(norm_name)
    stores['도로명키'] = stores[store_addr_col].apply(road_key)
    # 이름+도로명 건물번호가 모두 있을 때만 비교
    valid_card = card[(card['이름키'] != '') & (card['도로명키'] != '')]
    card_keys = set(zip(valid_card['이름키'], valid_card['도로명키']))
    stores['가맹점_동일키추정'] = [
        bool(n and a and (n, a) in card_keys)
        for n,a in zip(stores['이름키'], stores['도로명키'])
    ]
    stores['가맹여부_상태'] = stores['가맹점_동일키추정'].map({
        True:'기존가맹점_추정', False:'가맹여부_확인필요'
    })
    candidates = stores[stores['가맹여부_상태']=='가맹여부_확인필요'].copy()
    candidates['학교주변여부'] = '추후_거리분석필요'
    candidates['실제_비가맹확인'] = ''
    print('기존가맹점과 동일키 추정:', stores['가맹점_동일키추정'].sum())
    print('가맹 여부 확인 필요한 상가:', len(candidates))
    save_csv(candidates, os.path.join(B_DIR,'B_가맹여부_확인필요_상가.csv'))

기존가맹점과 동일키 추정: 2495
가맹 여부 확인 필요한 상가: 15489
저장: B_가맹여부_확인필요_상가.csv (15,489행)


## 8. B: 학교 주변 상가 후보 (직선거리 500m 예시)

이 단계는 **학교에서 반경 500m 이내의 상가 후보를 찾는 예비 탐색**이다. `학교 주변에 급식카드 가맹점이 없다`는 의미의 **사각지대 판정은 아니다.** 이를 판단하려면 가맹점의 좌표도 충분히 확인해야 한다.

- 거리는 위도·경도로 계산한 **직선거리**이며 실제 보행 거리나 도로 접근성은 다르다.
- 상가 업종의 식사 가능 여부와 가맹 여부도 별도로 검증해야 한다.
- 필요하면 `RADIUS_M` 값을 변경할 수 있다.

In [ ]:
import numpy as np

RADIUS_M = 500  # 학교 기준 예비 직선거리

def haversine_m(lat1, lon1, lat2, lon2):
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlat, dlon = lat2 - lat1, lon2 - lon1
    h = np.sin(dlat / 2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon / 2)**2
    return 6371000 * 2 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))

if 'candidates' not in globals():
    print('상가의 상호명/주소 컬럼을 먼저 확인해 주세요.')
else:
    valid_schools = schools.dropna(subset=['위도_분석','경도_분석']).copy()
    valid_stores = candidates.dropna(subset=['위도_분석','경도_분석']).copy()

    if len(valid_schools) == 0 or len(valid_stores) == 0:
        print('좌표가 충분하지 않아 학교 주변 거리 계산을 생략합니다.')
    else:
        school_name = next((x for x in ['학교명','학교이름'] if x in valid_schools.columns), None)
        valid_stores = valid_stores.copy()
        min_dist = np.full(len(valid_stores), np.inf)
        near_school = np.full(len(valid_stores), '', dtype=object)

        for _, school in valid_schools.iterrows():
            dist = haversine_m(
                valid_stores['위도_분석'].to_numpy(),
                valid_stores['경도_분석'].to_numpy(),
                float(school['위도_분석']), float(school['경도_분석'])
            )
            update = dist < min_dist
            min_dist[update] = dist[update]
            near_school[update] = str(school[school_name]) if school_name else str(school.name)

        valid_stores['가까운학교'] = near_school
        valid_stores['가까운학교_직선거리m'] = np.round(min_dist, 1)
        school_near = valid_stores[
            valid_stores['가까운학교_직선거리m'] <= RADIUS_M
        ].copy()
        school_near['사각지대_판정'] = '미판정_현재가맹점좌표확인필요'
        school_near['상가_적합성검토'] = ''
        print('학교 반경', RADIUS_M, 'm 이내 상가 후보:', len(school_near))
        save_csv(school_near, os.path.join(B_DIR, 'B_학교주변_상가_예비후보.csv'))


학교 반경 500 m 이내 상가 후보: 10538
저장: B_학교주변_상가_예비후보.csv (10,538행)


## 9. A: 사람이 검수한 결과 반영 (선택 사항)

**검수 전에는 실행하지 않아도 된다.** `A_사람검수_입력양식.csv`를 엑셀 등으로 열어 `검수판정`에 `표시가능`, `제외확정`, `보류` 중 하나를 기록하고 저장한 후 아래 셀을 실행한다.

`제외확정`에는 구체적인 확인 근거가 있어야 한다. 이 셀은 검수가 실제로 기록된 경우에만 반영 결과를 저장한다.

In [ ]:
path = os.path.join(A_DIR,'A_사람검수_입력양식.csv')
reviewed = pd.read_csv(path, encoding='utf-8-sig')
reviewed['검수판정'] = reviewed['검수판정'].fillna('').astype(str).str.strip()
known = {'표시가능','제외확정','보류',''}
unknown = reviewed[~reviewed['검수판정'].isin(known)]
if len(unknown):
    print('잘못 입력된 검수판정이 있습니다. 표시가능/제외확정/보류 중 선택하세요.')
elif not reviewed['검수판정'].isin(['표시가능','제외확정']).any():
    print('확정 판정이 없으므로 검증반영 지도 파일 생성은 생략합니다.')
else:
    assert reviewed['가맹점ID'].is_unique
    review_map = reviewed.set_index('가맹점ID')['검수판정']
    checked = card.copy()
    checked['검수판정'] = checked['가맹점ID'].map(review_map).fillna('')
    excluded = checked[checked['검수판정']=='제외확정'].copy()
    ready = checked[checked['검수판정']!='제외확정'].copy()
    save_csv(excluded, os.path.join(A_DIR,'A_제외확정_검수기록.csv'))
    save_csv(ready, os.path.join(A_DIR,'A_지도용_제외확정반영.csv'))
    print('주의: 지도용 파일의 보류/미연결 가맹점은 아직 적합성 검증이 완료되지 않았습니다.')


확정 판정이 없으므로 검증반영 지도 파일 생성은 생략합니다.


### A: 아동급식카드 지도
- **기준 목록**: `A_가맹점_전체정리.csv`
- **중복 의심**: `A_중복의심_검토.csv`
- **기타/모호한 업종**: `A_업종기타_재분류검토.csv`
- **주점 관련 분류**: `A_주점관련_검토후보.csv`
- **사람 검수 대상**: `A_사람검수_입력양식.csv`
- `A_지도용_검토상태포함.csv`는 **임시 상태표시용**이며, 최종 '안전한 지도'로 공개하기 전에 제외 여부를 검수해야 한다.

### B: 사각지대와 신규 가맹 추천
- `B_춘천학교_좌표준비.csv`, `B_춘천상가_좌표준비.csv`로 위치 분석을 시작할 수 있다.
- `B_가맹여부_확인필요_상가.csv`는 **비가맹점 확정 목록이 아니다.**
- `B_학교주변_상가_예비후보.csv`는 학교에서 500m 이내 직선거리 후보이며 **사각지대 확정 목록은 아니다.**
- 학교 근처의 **실제 서비스 사각지대**를 판단하려면 가맹점 위치 좌표 확보, 검증된 식사 가능 업소와 거리 계산, 지리적 접근성·영업시간 확인이 필요하다. 학교 반경은 예컨대 500m를 가정해 시범 분석할 수 있지만 결과에서 기준을 명시해야 한다.

**최종 주의:** 이전 197개는 자동 추출한 '주점 관련 업태 후보'이지 부적합 확정 가맹점 수가 아니다. 수작업으로 검증해 제외 확정하기 전에는 급식카드 목록에서 삭제하지 않는다.

## 10. 전처리 결과를 하나의 엑셀 파일로 정리

앞에서 생성한 **A(가맹점 지도용 점검)** 및 **B(학교 주변 상가 분석)** 결과 CSV를 한 파일에서 확인할 수 있도록 7개의 시트로 묶는다.

- **A 시트 4개**: 전체 가맹점, 중복 의심, 업종 재분류, 주점 관련 검토 후보
- **B 시트 3개**: 학교 위치, 가맹 여부 확인이 필요한 상가, 학교 주변 상가 후보
- 한글 파일명의 유니코드 표기 방식이 달라도 파일을 찾을 수 있도록 처리한다.

**결과 파일:** `MyDrive/data/데이터톤_전처리_통합.xlsx`

> 이 엑셀은 분석·검수용 통합 자료이며, 검토 후보를 부적합 가맹점 또는 비가맹점으로 확정한 결과는 아니다.

In [ ]:

import pandas as pd
import os
import unicodedata

folder = "/content/drive/MyDrive/data/processed_v2"
output = "/content/drive/MyDrive/data/데이터톤_전처리_통합.xlsx"

# 시트 이름 : 기존 CSV 파일명
files = {
    "A_전체가맹점": "A_지도/A_지도용_검토상태포함.csv",
    "A_중복의심": "A_지도/A_중복의심_검토.csv",
    "A_업종재분류": "A_지도/A_업종기타_재분류검토.csv",
    "A_주점검토": "A_지도/A_주점관련_검토후보.csv",
    "B_학교위치": "B_사각지대/B_춘천학교_좌표준비.csv",
    "B_가맹확인필요": "B_사각지대/B_가맹여부_확인필요_상가.csv",
    "B_학교주변상가": "B_사각지대/B_학교주변_상가_예비후보.csv"
}

# 한글 파일명 비교용 함수
def find_file(relative_path):
    target = unicodedata.normalize("NFC", relative_path)
    for root, dirs, filenames in os.walk(folder):
        for filename in filenames:
            path = os.path.join(root, filename)
            relative = os.path.relpath(path, folder)
            if unicodedata.normalize("NFC", relative) == target:
                return path
    raise FileNotFoundError(relative_path)

# 통합 엑셀 생성
with pd.ExcelWriter(output, engine="openpyxl") as writer:
    for sheet_name, filename in files.items():
        path = find_file(filename)

        try:
            df = pd.read_csv(path, encoding="utf-8-sig",
                             low_memory=False)
        except UnicodeDecodeError:
            df = pd.read_csv(path, encoding="cp949",
                             low_memory=False)

        df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(sheet_name, ":", len(df), "행 저장")

print("\n통합 엑셀 파일 생성 완료!")
print(output)


A_전체가맹점 : 4568 행 저장
A_중복의심 : 350 행 저장
A_업종재분류 : 414 행 저장
A_주점검토 : 197 행 저장
B_학교위치 : 76 행 저장
B_가맹확인필요 : 15489 행 저장
B_학교주변상가 : 10538 행 저장

통합 엑셀 파일 생성 완료!
/content/drive/MyDrive/data/데이터톤_전처리_통합.xlsx


## 11. A 가맹점 데이터 구조 확인

좌표를 연결하기 전에 `A_지도용_검토상태포함.csv`를 다시 읽어 **가맹점 수, 열 이름, 앞부분 데이터**를 확인한다.

이후 코드에서는 이 데이터를 `df`로 사용하며, 가맹점 이름과 도로명주소를 기준으로 B 상가정보와 비교한다.

In [ ]:
# A 가맹점 데이터 확인
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

# 기존에 만든 A 전처리 파일 불러오기
path = "/content/drive/MyDrive/data/processed_v2/A_지도/A_지도용_검토상태포함.csv"

df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)

print("전체 가맹점 수:", len(df))

print("\n전체 열 이름:")
for col in df.columns:
    print(col)

print("\n앞부분 데이터:")
display(df.head(5))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
전체 가맹점 수: 4568

전체 열 이름:
가맹점명
가맹점유형
업종명
시도명
시군구명
시군구코드
소재지도로명주소
관리기관명
관리기관전화번호
데이터기준일자
원본행번호
가맹점ID
이름키
주소정리
도로명키
지번키
상세주소키
위치키
중복의심
중복이유
공식_업태명
매칭상태
업종검토사유
상호명_분류단서
업태점검그룹
지도표시상태

앞부분 데이터:


,가맹점명,가맹점유형,업종명,시도명,시군구명,시군구코드,소재지도로명주소,관리기관명,관리기관전화번호,데이터기준일자,...,상세주소키,위치키,중복의심,중복이유,공식_업태명,매칭상태,업종검토사유,상호명_분류단서,업태점검그룹,지도표시상태
0,(뉴)진수성찬,일반가맹점,한식,강원특별자치도,춘천시,4181000,강원특별자치도 춘천시 공지로 450 1층,강원특별자치도 춘천시청,033-250-4043,2026-08-26,...,강원도춘천시공지로4501층,도로:공지로 450,False,NaN,한식,검증대기,NaN,NaN,NaN,기초표시후보
1,(사) 담배인삼공제회 (세인트콕스),일반가맹점,일반대중음식,강원특별자치도,춘천시,4181000,"강원 춘천시 스포츠타운길399번길 22, 1층 (삼천동)",강원특별자치도 춘천시청,033-250-4043,2026-08-26,...,"강원도춘천시스포츠타운길399번길22,1층(삼천동)",도로:스포츠타운길399번길 22,False,NaN,NaN,미연결,NaN,NaN,NaN,참고업태미연결
2,(신)옹달샘,일반가맹점,한식,강원특별자치도,춘천시,4181000,강원특별자치도 춘천시 영서로 2352-44 상가12동83호,강원특별자치도 춘천시청,033-250-4043,2026-08-26,...,강원도춘천시영서로2352-44상가12동83호,도로:영서로 2352-44,False,NaN,분식,검증대기,NaN,NaN,NaN,기초표시후보
3,(유)아웃백스테이크하우스 춘천점,일반가맹점,양식,강원특별자치도,춘천시,4181000,강원도 춘천시 후석로 120 2층(석사동),강원특별자치도 춘천시청,033-250-4043,2026-08-26,...,강원도춘천시후석로1202층(석사동),도로:후석로 120,False,NaN,NaN,미연결,NaN,NaN,NaN,참고업태미연결
4,(주) 아워홈 더플레이어스GC점,일반가맹점,한식,강원특별자치도,춘천시,4181000,강원도 춘천시 동산면 새술막길 438 1층,강원특별자치도 춘천시청,033-250-4043,2026-08-26,...,강원도춘천시동산면새술막길4381층,도로:새술막길 438,False,NaN,NaN,미연결,NaN,NaN,NaN,참고업태미연결


## 12. B 상가정보의 좌표 확인

`B_춘천상가_좌표준비.csv`를 불러와 **상가 수, 열 이름, 위도·경도 등 데이터 구조**를 확인한다.

이후에는 이 데이터를 `df_b`로 사용해 A 가맹점 목록에 좌표를 연결할 후보 자료로 활용한다. 상가 목록에 존재한다는 사실만으로 아동급식카드 가맹점 여부가 확인되는 것은 아니다.

In [ ]:
# B 상가정보의 좌표 확인
# B에서 전처리한 춘천시 상가정보 불러오기
path_b = "/content/drive/MyDrive/data/processed_v2/B_사각지대/B_춘천상가_좌표준비.csv"

df_b = pd.read_csv(path_b, encoding="utf-8-sig", low_memory=False)

print("전체 상가 수:", len(df_b))

print("\n전체 열 이름:")
for col in df_b.columns:
    print(col)

print("\n앞부분 데이터:")
display(df_b.head(3))


전체 상가 수: 17984

전체 열 이름:
상가업소번호
상호명
지점명
상권업종대분류코드
상권업종대분류명
상권업종중분류코드
상권업종중분류명
상권업종소분류코드
상권업종소분류명
표준산업분류코드
표준산업분류명
시도코드
시도명
시군구코드
시군구명
행정동코드
행정동명
법정동코드
법정동명
지번코드
대지구분코드
대지구분명
지번본번지
지번부번지
지번주소
도로명코드
도로명
건물본번지
건물부번지
건물관리번호
건물명
도로명주소
구우편번호
신우편번호
동정보
층정보
호정보
경도
위도
위도_분석
경도_분석

앞부분 데이터:


,상가업소번호,상호명,지점명,상권업종대분류코드,상권업종대분류명,상권업종중분류코드,상권업종중분류명,상권업종소분류코드,상권업종소분류명,표준산업분류코드,...,도로명주소,구우편번호,신우편번호,동정보,층정보,호정보,경도,위도,위도_분석,경도_분석
0,MA0106202201A1902607,파크랜드춘천,명동점,G2,소매,G213,오락용품 소매,G21304,운동용품 소매업,G47631,...,강원특별자치도 춘천시 금강로 80,200941,24353,NaN,NaN,NaN,127.729466,37.878276,37.878276,127.729466
1,MA0106202201A0153998,파파게노춘천명동,NaN,G2,소매,G209,섬유·의복·신발 소매,G20905,기타 의류 소매업,G47419,...,강원특별자치도 춘천시 금강로 63,200941,24352,NaN,NaN,NaN,127.728787,37.879685,37.879685,127.728787
2,MA0106202201A2298047,올리브영춘천,후평점,G2,소매,G221,기타 상품 소매,G22199,그 외 기타 상품 전문 소매업,G47859,...,강원특별자치도 춘천시 후석로 298-1,200956,24304,NaN,NaN,NaN,127.751548,37.878011,37.878011,127.751548


## 13. 가맹점명과 주소를 이용한 1차 좌표 연결

A 가맹점 목록과 B 상가 목록을 비교해 좌표가 연결될 가능성이 높은 행을 찾는다.

1. **상호명 정규화**: 공백·특수문자를 제외한 비교용 이름을 만든다.
2. **주소 정리**: 도로명과 건물번호를 추출해 비교용 주소를 만든다.
3. **중복 후보 제외**: B에서 같은 이름과 주소를 가진 행이 여러 개면 자동 연결 대상으로 사용하지 않는다.
4. **이름 + 주소 동시 일치**: 두 비교값이 일치하는 경우 B의 위도·경도를 A에 연결한다.
5. **결과 확인**: 좌표 연결 후보 수, 미연결 수, 연결 비율을 출력한다.

> 이 단계의 결과는 **이름과 주소가 일치한 좌표 연결 후보**이다. 같은 상호·건물이라고 해도 실제 점포 위치가 정확한지는 별도 검증이 필요하다.

In [ ]:
# 같은 가맹점 확인
import re

# 기존 데이터 복사
a = df.copy()
b = df_b.copy()

# 1. 가게 이름 정리
def clean_name(name):
    if pd.isna(name):
        return ""
    return re.sub(r"[^가-힣a-z0-9]", "", str(name).lower())

# 2. 도로명과 건물번호 추출
def clean_address(address):
    if pd.isna(address):
        return ""

    address = str(address)

    match = re.search(
        r"([가-힣0-9]+(?:로|길))\s*(\d+(?:-\d+)?)",
        address
    )

    if match:
        return match.group(1) + " " + match.group(2)

    return ""

# 3. 비교용 열 생성
a["비교이름"] = a["가맹점명"].apply(clean_name)
b["비교이름"] = b["상호명"].apply(clean_name)

a["비교주소"] = a["소재지도로명주소"].apply(clean_address)
b["비교주소"] = b["도로명주소"].apply(clean_address)

# 4. 유효한 이름과 주소가 있는 B 데이터만 사용
b_valid = b[
    (b["비교이름"] != "") &
    (b["비교주소"] != "")
].copy()

# 이름과 주소가 같은 B 상가가 여러 개면 일단 제외
b_valid = b_valid.drop_duplicates(
    subset=["비교이름", "비교주소"],
    keep=False
)

# 5. A와 B 연결
matched = a.merge(
    b_valid[["비교이름", "비교주소", "위도", "경도"]],
    on=["비교이름", "비교주소"],
    how="left",
    validate="m:1"
)

# 6. 결과 확인
success = matched["위도"].notna() & matched["경도"].notna()

print("전체 가맹점:", len(matched))
print("좌표 연결 후보:", success.sum())
print("좌표 미연결:", (~success).sum())
print("연결 비율:", round(success.mean() * 100, 2), "%")

print("\n연결된 가맹점 예시:")
display(
    matched.loc[
        success,
        ["가맹점명", "소재지도로명주소", "위도", "경도"]
    ].head(10)
)


전체 가맹점: 4568
좌표 연결 후보: 2564
좌표 미연결: 2004
연결 비율: 56.13 %

연결된 가맹점 예시:


,가맹점명,소재지도로명주소,위도,경도
30,102국밥,강원도 춘천시 신북읍 영서로 3278 .,37.941318,127.712513
34,1단지 김밥,강원특별자치도 춘천시 춘천로310번길 24 1층(후평동),37.880421,127.746560
39,29페이지,"강원도 춘천시 둥지길 18-9 1,2층",37.874098,127.736324
40,2린닭갈비,"강원도 춘천시 서부대성로239번길 47-26 ,2층",37.876008,127.746220
43,33떡볶이&꼬마김밥 춘천우두점,강원특별자치도 춘천시 충열로 10-14 1층,37.900379,127.728945
45,33떡볶이춘천거두점,강원도 춘천시 동내면 춘천순환로 66 1층,37.855273,127.751317
47,5.5닭갈비,강원 춘천시 남춘로 33 (퇴계동),37.864220,127.731089
49,59쌀피자,강원특별자치도 춘천시 충열로 39 a동 108호,37.899490,127.732734
50,5감족발,강원도 춘천시 효평길 43 1층,37.878385,127.740834
52,60계강원춘천우두점,강원특별자치도 춘천시 충열로16번길 5-15 .,37.899417,127.730039


## 14. 1차 좌표 연결 결과 저장

1차 연결 결과를 CSV로 따로 보관해 다음 단계에서 추가 연결 여부를 확인할 수 있게 한다.

- `A_가맹점_좌표연결_중간결과.csv`: 좌표가 연결된 행과 연결되지 않은 행을 포함한 전체 중간 결과
- `A_좌표미연결_가맹점.csv`: 1차 연결에서 위도 또는 경도를 확보하지 못한 가맹점

**저장 위치:** `MyDrive/data/processed_v2/A_지도/`

In [ ]:
# 좌표 연결 결과 저장하기

import os

folder = "/content/drive/MyDrive/data/processed_v2/A_지도"

os.makedirs(folder, exist_ok=True)

# 1. 연결 결과 전체 저장
matched.to_csv(
    folder + "/A_가맹점_좌표연결_중간결과.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. 좌표가 없는 가맹점만 저장
unmatched = matched[
    matched["위도"].isna() | matched["경도"].isna()
].copy()

unmatched.to_csv(
    folder + "/A_좌표미연결_가맹점.csv",
    index=False,
    encoding="utf-8-sig"
)

print("전체 가맹점:", len(matched))
print("좌표 미연결:", len(unmatched))
print("파일 저장 완료!")


전체 가맹점: 4568
좌표 미연결: 2004
파일 저장 완료!


## 15. 동일 건물 주소를 이용한 2차 좌표 연결

1차에서 이름과 주소가 함께 일치하지 않아 연결하지 못한 가맹점에 대해, **같은 건물의 주소 정보**를 추가로 활용한다.

- B 상가정보에서 주소와 좌표가 있는 행을 선택한다.
- 같은 건물에 연결된 좌표들의 **중앙값**을 대표 좌표 후보로 계산한다.
- 건물 내 좌표 차이가 코드에서 설정한 범위를 넘는 경우에는 해당 건물을 추가 연결 대상에서 제외한다.
- 기존에 좌표가 없는 가맹점에만 이 대표 좌표를 채운다.
- `좌표출처` 열로 `기존_이름주소매칭`, `동일건물_위치추정`, `미연결`을 구분한다.

> **주의:** `동일건물_위치추정`은 해당 가맹점의 정확한 출입구나 점포 위치를 확인한 값이 아니라 건물 단위의 추정 좌표이다. 코드에 표시된 `기존 연결: 2564`는 고정된 출력 문구이므로, 실제 연결 결과는 계산된 열과 최종 출력값으로 확인해야 한다.

In [ ]:
# 주소만으로 추가 좌표 찾기
import pandas as pd

# B 상가정보에서 주소와 좌표가 있는 데이터만 선택
b_address = b[
    (b["비교주소"] != "") &
    b["위도"].notna() &
    b["경도"].notna()
].copy()

# 같은 건물의 좌표가 서로 크게 다른 경우 제외
def check_building(group):
    lat = group["위도"].median()
    lon = group["경도"].median()

    # 대략 50m 이내의 좌표만 허용
    lat_diff = (group["위도"] - lat).abs().max()
    lon_diff = (group["경도"] - lon).abs().max()

    if lat_diff > 0.00045 or lon_diff > 0.00057:
        return None

    return pd.Series({"추가위도": lat, "추가경도": lon})

building_coords = (
    b_address.groupby("비교주소")
    .apply(check_building)
    .dropna()
    .reset_index()
)

# 기존에 연결하지 못한 가맹점만 추가 연결
result = matched.merge(
    building_coords,
    on="비교주소",
    how="left",
    validate="m:1"
)

# 기존 좌표가 없을 때만 건물 좌표 사용
result["좌표출처"] = "기존_이름주소매칭"

missing = result["위도"].isna() | result["경도"].isna()

new_match = (
    missing &
    result["추가위도"].notna() &
    result["추가경도"].notna()
)

result.loc[new_match, "위도"] = result.loc[new_match, "추가위도"]
result.loc[new_match, "경도"] = result.loc[new_match, "추가경도"]
result.loc[new_match, "좌표출처"] = "동일건물_위치추정"

result.loc[
    result["위도"].isna() | result["경도"].isna(),
    "좌표출처"
] = "미연결"

print("기존 연결:", 2564)
print("추가 연결:", new_match.sum())
print("최종 좌표 확보:", result["위도"].notna().sum())
print("남은 미연결:", result["위도"].isna().sum())


기존 연결: 2564
추가 연결: 1560
최종 좌표 확보: 4124
남은 미연결: 444


/tmp/ipykernel_989/4241442847.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(check_building)


## 16. 최종 좌표 통합 파일 저장

1차 이름·주소 매칭과 2차 동일 건물 주소 추정 결과를 합쳐 **지도 연계용 좌표 후보 데이터**를 저장한다.

- 임시 계산 열(`추가위도`, `추가경도`)을 제거한다.
- 위도·경도가 코드에 설정된 범위를 벗어난 행은 `좌표확인필요`로 표시한다.
- 좌표가 있는 전체 결과와 미연결 가맹점 목록을 별도의 CSV로 저장한다.

**생성 파일**
- `A_가맹점_좌표통합.csv`: 전체 가맹점 및 좌표 연결·출처·확인 상태
- `A_좌표미연결_최종후보.csv`: 최종 단계에서도 좌표를 연결하지 못한 행

**저장 위치:** `MyDrive/data/processed_v2/A_지도/`

In [ ]:
# 좌표 연결 결과 저장
import os

folder = "/content/drive/MyDrive/data/processed_v2/A_지도"
os.makedirs(folder, exist_ok=True)

# 지도에 사용할 좌표 데이터 정리
final_a = result.copy()

# 임시로 생성한 추가 좌표 열 제거
final_a = final_a.drop(
    columns=["추가위도", "추가경도"],
    errors="ignore"
)

# 좌표 범위가 비정상적인 데이터는 확인 대상으로 표시
final_a["좌표확인필요"] = (
    final_a["위도"].notna() &
    final_a["경도"].notna() &
    ~(
        final_a["위도"].between(37, 39) &
        final_a["경도"].between(127, 129)
    )
)

# 전체 결과 저장
final_a.to_csv(
    folder + "/A_가맹점_좌표통합.csv",
    index=False,
    encoding="utf-8-sig"
)

# 좌표가 없는 데이터 별도 저장
no_coords = final_a[
    final_a["위도"].isna() |
    final_a["경도"].isna()
]

no_coords.to_csv(
    folder + "/A_좌표미연결_최종후보.csv",
    index=False,
    encoding="utf-8-sig"
)

print("전체 가맹점:", len(final_a))
print("좌표 확보:", final_a["위도"].notna().sum())
print("좌표 미확보:", len(no_coords))
print("좌표 범위 확인 필요:", final_a["좌표확인필요"].sum())
print("저장 완료!")


전체 가맹점: 4568
좌표 확보: 4124
좌표 미확보: 444
좌표 범위 확인 필요: 0
저장 완료!


---
## 결과 해석 및 후속 작업

이 노트북은 가맹점의 중복·업종을 **검토할 후보**를 추리고, 학교·상가 정보와 비교하여 **좌표 연결 후보**를 확보하는 데 목적이 있다.

- `중복의심`·`주점관련_보류` 등은 자동 제외 대상이 아니며, 실제 영업 형태와 급식카드 이용 가능 여부를 확인해야 한다.
- `동일건물_위치추정` 좌표와 범위 검사를 통과한 좌표도 실제 매장 위치가 검증됐다는 뜻은 아니다.
- 학교 반경 500m 분석은 **직선거리 기반 예비 탐색**이며 실제 보행 경로나 아동급식카드 사각지대의 확정 판정이 아니다.
- 서비스 지도에 적용하기 전 **미연결 좌표, 동일 건물 추정 좌표, 검수 보류 업종**을 추가 확인하는 것이 필요하다.
